In [70]:
import duckdb
import leafmap
import pandas as pd

In [71]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [72]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [73]:
# Connect to DuckDB in-memory database
con = duckdb.connect()
# Connect to DuckDB file-based database
# con = duckdb.connect('filename.db')
# Connect to DuckDB in-memory database using SQL magic
%sql duckdb:///:memory:

# the %%sql commands don't require strings, whereas the duckdb.connect() requires strings, making the magic commands a lot more convenient

In [74]:
con.install_extension("httpfs")
con.load_extension("httpfs")
con.install_extension("spatial")
con.load_extension("spatial")

## View all extensions: con.sql("FROM duckdb_extensions();")
# this will give a DuckDB output with only first 10 and last 10 rows
# to convert to DF, use con.sql("FROM duckdb_extensions();").df() at the end of the command

In [75]:
con.sql("FROM duckdb_extensions();")

┌──────────────────┬─────────┬───────────┬─────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────┬───────────────────┬───────────────────┬───────────────────┬────────────────┐
│  extension_name  │ loaded  │ installed │                                  install_path                                   │                                    description                                     │      aliases      │ extension_version │   install_mode    │ installed_from │
│     varchar      │ boolean │  boolean  │                                     varchar                                     │                                      varchar                                       │     varchar[]     │      varchar      │      varchar      │    varchar     │
├──────────────────┼─────────┼───────────┼─────────────────────────────────────────────────────────────────────────────────┼───────────────

In [76]:
#%%sql
#INSTALL httpfs;
#LOAD httpfs;
#INSTALL spatial;
#LOAD spatial;

In [77]:
#%%sql
#FROM duckdb_extensions();

# this will automatically convert to pandas DF because of the earlier config setting

In [78]:
#%%sql
#SELECT * FROM 'https://opengeos.org/data/duckdb/cities.csv'

In [79]:
#%%sql
#SELECT * FROM 'https://opengeos.org/data/duckdb/countries.csv'

In [80]:
# persistent tables - downloaded from the URL and store in the cache so that we don't have to keep using the URL every time

#%%sql
#CREATE OR REPLACE TABLE cities AS SELECT * FROM 'https://opengeos.org/data/duckdb/cities.csv';

In [81]:
#%%sql
#CREATE OR REPLACE TABLE countries AS SELECT * FROM 'https://opengeos.org/data/duckdb/countries.csv';

In [82]:
con.sql("CREATE OR REPLACE TABLE cities AS SELECT * FROM 'https://opengeos.org/data/duckdb/cities.csv';")

In [83]:
con.sql("CREATE OR REPLACE TABLE countries AS SELECT * FROM 'https://opengeos.org/data/duckdb/countries.csv';")

In [84]:
con.sql("SELECT COUNT(*) FROM cities;")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1249 │
└──────────────┘

In [85]:
con.sql("SELECT COUNT(DISTINCT country) FROM cities;")

┌─────────────────────────┐
│ count(DISTINCT country) │
│          int64          │
├─────────────────────────┤
│                     200 │
└─────────────────────────┘

In [86]:
con.sql("SELECT MIN(population) FROM cities;")

┌─────────────────┐
│ min(population) │
│      int64      │
├─────────────────┤
│               0 │
└─────────────────┘

In [87]:
con.sql("SELECT AVG(population) FROM cities;")

┌────────────────────┐
│  avg(population)   │
│       double       │
├────────────────────┤
│ 1181372.6989591673 │
└────────────────────┘

In [88]:
con.sql("SELECT MAX(population) FROM cities;")

┌─────────────────┐
│ max(population) │
│      int64      │
├─────────────────┤
│        35676000 │
└─────────────────┘

In [89]:
con.sql("SELECT * FROM cities WHERE population > 1000000 ORDER BY population DESC LIMIT 10;")

┌───────┬──────────────┬─────────┬───────────┬───────────┬────────────┐
│  id   │     name     │ country │ latitude  │ longitude │ population │
│ int64 │   varchar    │ varchar │  double   │  double   │   int64    │
├───────┼──────────────┼─────────┼───────────┼───────────┼────────────┤
│  1240 │ Tokyo        │ JPN     │  35.68502 │ 139.75141 │   35676000 │
│  1225 │ New York     │ USA     │  40.74998 │ -73.98002 │   19040000 │
│  1231 │ Mexico City  │ MEX     │  19.44244 │ -99.13099 │   19028000 │
│  1241 │ Mumbai       │ IND     │  19.01699 │  72.85699 │   18978000 │
│  1246 │ Sao Paulo    │ BRA     │ -23.55868 │ -46.62502 │   18845000 │
│  1149 │ Delhi        │ IND     │  28.66999 │     77.23 │   15926000 │
│  1239 │ Shanghai     │ CHN     │  31.21645 │  121.4365 │   14987000 │
│  1244 │ Kolkata      │ IND     │  22.49497 │  88.32468 │   14787000 │
│  1176 │ Dhaka        │ BGD     │  23.72306 │  90.40858 │   12797394 │
│  1218 │ Buenos Aires │ ARG     │  -34.6025 │ -58.39753 │   127

In [90]:
con.sql("SELECT country, COUNT(*) as city_count, AVG(population) as avg_population FROM cities GROUP BY country ORDER BY city_count DESC LIMIT 10;").df()

,country,city_count,avg_population
0,USA,114,1.372504e+06
1,CHN,100,2.171320e+06
2,RUS,81,5.840291e+05
3,IND,69,2.243626e+06
4,BRA,46,1.756396e+06
5,CAN,45,3.860355e+05
6,ATA,40,9.467500e+01
7,AUS,36,4.566392e+05
8,FRA,30,6.735245e+05
9,MEX,27,1.637853e+06


In [91]:
# output results in lists and tuples

con.sql('SELECT 42').fetchall()

[(42,)]

In [92]:
# output results in pandas DataFrame

con.sql('SELECT 42').df()

,42
0,42


In [93]:
# output in numpy arrays

con.sql('SELECT 42').fetchnumpy()

{'42': array([42], dtype=int32)}

In [94]:
url = "https://opengeos.org/data/duckdb/cities.zip"
leafmap.download_file(url, unzip=True)

cities.zip already exists. Skip downloading. Set overwrite=True to overwrite.


'd:\\DevSource\\sgsup-arjav-322\\cities.zip'

In [95]:
con.read_csv('cities.csv', parallel=True)

# parallel is for larger files - uses multiple threads to read the file faster
# but for smaller files it may not make a difference and can even be slower due to overhead of managing threads

┌───────┬──────────────────┬─────────┬───────────┬───────────┬────────────┐
│  id   │       name       │ country │ latitude  │ longitude │ population │
│ int64 │     varchar      │ varchar │  double   │  double   │   int64    │
├───────┼──────────────────┼─────────┼───────────┼───────────┼────────────┤
│     1 │ Bombo            │ UGA     │    0.5833 │   32.5333 │      75000 │
│     2 │ Fort Portal      │ UGA     │     0.671 │    30.275 │      42670 │
│     3 │ Potenza          │ ITA     │    40.642 │    15.799 │      69060 │
│     4 │ Campobasso       │ ITA     │    41.563 │    14.656 │      50762 │
│     5 │ Aosta            │ ITA     │    45.737 │     7.315 │      34062 │
│     6 │ Mariehamn        │ ALD     │    60.097 │    19.949 │      10682 │
│     7 │ Ramallah         │ PSE     │  31.90294 │  35.20621 │      24599 │
│     8 │ Vatican City     │ VAT     │  41.90001 │  12.44781 │        832 │
│     9 │ Poitier          │ FRA     │  46.58329 │   0.33328 │      85960 │
│    10 │ Cl

In [96]:
# you can also do con.sql("SELECT * FROM 'cities.csv'")

In [97]:
con.read_parquet('cities.parquet')

┌─────────┬─────────────────────────────┬────────┬───────────┬───────────┬──────────────────┬────────────┐
│ country │          geometry           │   id   │ latitude  │ longitude │       name       │ population │
│ varchar │          geometry           │ double │  double   │  double   │     varchar      │   double   │
├─────────┼─────────────────────────────┼────────┼───────────┼───────────┼──────────────────┼────────────┤
│ UGA     │ POINT (32.5333 0.5833)      │    1.0 │    0.5833 │   32.5333 │ Bombo            │    75000.0 │
│ UGA     │ POINT (30.275 0.671)        │    2.0 │     0.671 │    30.275 │ Fort Portal      │    42670.0 │
│ ITA     │ POINT (15.799 40.642)       │    3.0 │    40.642 │    15.799 │ Potenza          │    69060.0 │
│ ITA     │ POINT (14.656 41.563)       │    4.0 │    41.563 │    14.656 │ Campobasso       │    50762.0 │
│ ITA     │ POINT (7.315 45.737)        │    5.0 │    45.737 │     7.315 │ Aosta            │    34062.0 │
│ ALD     │ POINT (19.949 60.097)    

In [98]:
con.read_json('cities.json')

┌───────┬──────────────────┬─────────┬───────────┬───────────┬────────────┐
│  id   │       name       │ country │ latitude  │ longitude │ population │
│ int64 │     varchar      │ varchar │  double   │  double   │   int64    │
├───────┼──────────────────┼─────────┼───────────┼───────────┼────────────┤
│     1 │ Bombo            │ UGA     │    0.5833 │   32.5333 │      75000 │
│     2 │ Fort Portal      │ UGA     │     0.671 │    30.275 │      42670 │
│     3 │ Potenza          │ ITA     │    40.642 │    15.799 │      69060 │
│     4 │ Campobasso       │ ITA     │    41.563 │    14.656 │      50762 │
│     5 │ Aosta            │ ITA     │    45.737 │     7.315 │      34062 │
│     6 │ Mariehamn        │ ALD     │    60.097 │    19.949 │      10682 │
│     7 │ Ramallah         │ PSE     │  31.90294 │  35.20621 │      24599 │
│     8 │ Vatican City     │ VAT     │  41.90001 │  12.44781 │        832 │
│     9 │ Poitier          │ FRA     │  46.58329 │   0.33328 │      85960 │
│    10 │ Cl

In [99]:
con.sql("SELECT * FROM ST_Read('cities.geojson');")

┌───────┬──────────────────┬─────────┬───────────┬───────────┬────────────┬─────────────────────────────┐
│  id   │       name       │ country │ latitude  │ longitude │ population │            geom             │
│ int32 │     varchar      │ varchar │  double   │  double   │   int32    │          geometry           │
├───────┼──────────────────┼─────────┼───────────┼───────────┼────────────┼─────────────────────────────┤
│     1 │ Bombo            │ UGA     │    0.5833 │   32.5333 │      75000 │ POINT (32.5333 0.5833)      │
│     2 │ Fort Portal      │ UGA     │     0.671 │    30.275 │      42670 │ POINT (30.275 0.671)        │
│     3 │ Potenza          │ ITA     │    40.642 │    15.799 │      69060 │ POINT (15.799 40.642)       │
│     4 │ Campobasso       │ ITA     │    41.563 │    14.656 │      50762 │ POINT (14.656 41.563)       │
│     5 │ Aosta            │ ITA     │    45.737 │     7.315 │      34062 │ POINT (7.315 45.737)        │
│     6 │ Mariehamn        │ ALD     │    60.0

In [100]:
con.sql("SELECT * FROM ST_Read('cities.shp');")

┌───────┬──────────────────┬─────────┬───────────┬───────────┬────────────┬─────────────────────────────┐
│  id   │       name       │ country │ latitude  │ longitude │ population │            geom             │
│ int64 │     varchar      │ varchar │  double   │  double   │   int64    │          geometry           │
├───────┼──────────────────┼─────────┼───────────┼───────────┼────────────┼─────────────────────────────┤
│     1 │ Bombo            │ UGA     │    0.5833 │   32.5333 │      75000 │ POINT (32.5333 0.5833)      │
│     2 │ Fort Portal      │ UGA     │     0.671 │    30.275 │      42670 │ POINT (30.275 0.671)        │
│     3 │ Potenza          │ ITA     │    40.642 │    15.799 │      69060 │ POINT (15.799 40.642)       │
│     4 │ Campobasso       │ ITA     │    41.563 │    14.656 │      50762 │ POINT (14.656 41.563)       │
│     5 │ Aosta            │ ITA     │    45.737 │     7.315 │      34062 │ POINT (7.315 45.737)        │
│     6 │ Mariehamn        │ ALD     │    60.0

In [101]:
con.sql("""
CREATE TABLE IF NOT EXISTS cities AS
FROM 'https://opengeos.org/data/duckdb/cities.parquet'
""")

In [102]:
con.sql("COPY cities TO 'citiesW.csv' (HEADER, DELIMITER ',')")

In [103]:
con.sql("COPY (SELECT * FROM cities WHERE country='USA') to 'cities_us.json'")

In [104]:
con.sql("COPY (SELECT * FROM cities WHERE country='IND') TO 'cities_ind.parquet' (FORMAT PARQUET)")

In [105]:
con.sql("COPY (SELECT * FROM cities WHERE country='JPN') TO 'cities_jpn.geojson' WITH (FORMAT GDAL, DRIVER 'GeoJSON')")

In [107]:
con.sql("COPY (SELECT * FROM cities WHERE country='ITA') TO 'cities_ita.shp' WITH (FORMAT GDAL, DRIVER 'ESRI Shapefile')")

In [108]:
con.sql("COPY (SELECT * FROM cities WHERE country='BRA') TO 'cities_bra.gpkg' WITH (FORMAT GDAL, DRIVER 'GPKG')")

## NYC Data

In [109]:
url = "https://opengeos.org/data/duckdb/nyc_data.db.zip"
leafmap.download_file(url, unzip=True)

nyc_data.db.zip already exists. Skip downloading. Set overwrite=True to overwrite.


'd:\\DevSource\\sgsup-arjav-322\\nyc_data.db.zip'

In [110]:
con=duckdb.connect("nyc_data.db")

In [111]:
con.install_extension("spatial")
con.load_extension("spatial")

In [112]:
con.sql("SELECT name, ST_AsText(geom) FROM nyc_subway_stations LIMIT 10;")

┌──────────────┬─────────────────────────────────────────────┐
│     NAME     │               st_astext(geom)               │
│   varchar    │                   varchar                   │
├──────────────┼─────────────────────────────────────────────┤
│ Cortlandt St │ POINT (583521.854408956 4507077.862599085)  │
│ Rector St    │ POINT (583324.4866324601 4506805.373160211) │
│ South Ferry  │ POINT (583304.1823994748 4506069.654048115) │
│ 138th St     │ POINT (590250.10594797 4518558.019924332)   │
│ 149th St     │ POINT (590454.7399891173 4519145.719617855) │
│ 149th St     │ POINT (590465.8934191109 4519168.697483203) │
│ 161st St     │ POINT (590573.169495527 4520214.766177284)  │
│ 167th St     │ POINT (591252.8314104103 4520950.353355553) │
│ 167th St     │ POINT (590946.3972262995 4521077.318976877) │
│ 170th St     │ POINT (591583.6111452815 4521434.846626811) │
├──────────────┴─────────────────────────────────────────────┤
│ 10 rows                                          2 co

In [113]:
con.sql("SHOW TABLES;")

┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ nyc_census_blocks   │
│ nyc_homicides       │
│ nyc_neighborhoods   │
│ nyc_streets         │
│ nyc_subway_stations │
└─────────────────────┘

In [114]:
con.sql("SHOW VARIABLES; FROM nyc_subway_stations;")

┌──────────┬────────┬───────────────────┬─────────────────┬─────────────────┬─────────────────────────────────────────┬───────────────────────────────────┬───────────┬─────────┬─────────┬───────────┬──────────────┬─────────┬─────────┬──────────────────────────────────────────────┐
│ OBJECTID │   ID   │       NAME        │    ALT_NAME     │    CROSS_ST     │                LONG_NAME                │               LABEL               │  BOROUGH  │ NGHBHD  │ ROUTES  │ TRANSFERS │    COLOR     │ EXPRESS │ CLOSED  │                     geom                     │
│  double  │ double │      varchar      │     varchar     │     varchar     │                 varchar                 │              varchar              │  varchar  │ varchar │ varchar │  varchar  │   varchar    │ varchar │ varchar │                   geometry                   │
├──────────┼────────┼───────────────────┼─────────────────┼─────────────────┼─────────────────────────────────────────┼───────────────────────────────────

In [115]:
con.sql("SELECT name, boroname FROM nyc_neighborhoods WHERE ST_Intersects(geom, ST_GeomFromText('POINT(583571 4506714)'))")

┌────────────────────┬───────────┐
│        NAME        │ BORONAME  │
│      varchar       │  varchar  │
├────────────────────┼───────────┤
│ Financial District │ Manhattan │
└────────────────────┴───────────┘

In [116]:
con.sql("SELECT name, geom FROM nyc_subway_stations WHERE name = 'Broad St';")

┌──────────┬─────────────────────────────────────────────┐
│   NAME   │                    geom                     │
│ varchar  │                  geometry                   │
├──────────┼─────────────────────────────────────────────┤
│ Broad St │ POINT (583571.9059213118 4506714.341192182) │
└──────────┴─────────────────────────────────────────────┘

In [117]:
con.sql("SELECT COUNT(*) as nearby_stations FROM nyc_subway_stations WHERE ST_DWithin(geom, ST_GeomFromText('POINT(583571 4506714)'), 500);")

┌─────────────────┐
│ nearby_stations │
│      int64      │
├─────────────────┤
│              12 │
└─────────────────┘

In [118]:
con.sql("SELECT subways.name AS subway_name, neighborhoods.name AS neighborhood_name, neighborhoods.boroname AS borough FROM nyc_neighborhoods AS neighborhoods JOIN nyc_subway_stations AS subways ON ST_Intersects(neighborhoods.geom, subways.geom) WHERE subways.name = 'Broad St';")

┌─────────────┬────────────────────┬───────────┐
│ subway_name │ neighborhood_name  │  borough  │
│   varchar   │      varchar       │  varchar  │
├─────────────┼────────────────────┼───────────┤
│ Broad St    │ Financial District │ Manhattan │
└─────────────┴────────────────────┴───────────┘

In [119]:
con.sql("DESCRIBE nyc_subway_stations;")

# null = is it allowed as null
# key is null bc no primary key
# default is the automatic fallback value for new rows
# extra is any additional rules for that column

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ OBJECTID    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ID          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ NAME        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ALT_NAME    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ CROSS_ST    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ LONG_NAME   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ LABEL       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ BOROUGH     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ NGHBHD      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ROUTES      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ TRANSFERS   │ VARC

In [120]:
con.sql("SELECT DISTINCT COLOR from nyc_subway_stations;").df()

,COLOR
0,BLUE-GREY
1,CLOSED
2,BLUE-BROWN
3,PURPLE
4,GREY-YELLOW
5,BROWN-ORANGE-YELLOW
6,SI-BLUE
7,LIME
8,BLUE-ORANGE
9,ORANGE-LIME


In [121]:
con.sql("SELECT subways.name AS subway_name, neighborhoods.name AS neighborhood_name, neighborhoods.boroname AS borough FROM nyc_neighborhoods AS neighborhoods JOIN nyc_subway_stations AS subways ON ST_Intersects(neighborhoods.geom, subways.geom) WHERE subways.color = 'Red';")

┌─────────────┬───────────────────┬─────────┐
│ subway_name │ neighborhood_name │ borough │
│   varchar   │      varchar      │ varchar │
├─────────────┴───────────────────┴─────────┤
│                  0 rows                   │
└───────────────────────────────────────────┘

In [122]:
con.sql("DESCRIBE nyc_census_blocks")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ BLKID       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ POPN_TOTAL  │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ POPN_WHITE  │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ POPN_BLACK  │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ POPN_NATIV  │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ POPN_ASIAN  │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ POPN_OTHER  │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ BORONAME    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ geom        │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [123]:
con.sql("""
SELECT
        100*SUM(popn_white)/SUM(popn_total) AS pct_white,
        100*SUM(popn_black)/SUM(popn_total) AS pct_black,
        100*SUM(popn_asian)/SUM(popn_total) AS pct_asian,
        100*SUM(popn_nativ)/SUM(popn_total) AS pct_native,
        SUM(popn_total) AS total_population
    FROM nyc_census_blocks;""")

┌───────────────────┬────────────────────┬───────────────────┬────────────────────┬──────────────────┐
│     pct_white     │     pct_black      │     pct_asian     │     pct_native     │ total_population │
│      double       │       double       │      double       │       double       │      int128      │
├───────────────────┼────────────────────┼───────────────────┼────────────────────┼──────────────────┤
│ 44.00395007628105 │ 25.546578900241613 │ 12.70171174865126 │ 0.7035079495713289 │          8175032 │
└───────────────────┴────────────────────┴───────────────────┴────────────────────┴──────────────────┘

In [124]:
con.sql("""
    SELECT DISTINCT routes FROM nyc_subway_stations AS subways,
    WHERE strpos(subways.routes,'A') > 0;
    """)

┌─────────┐
│ ROUTES  │
│ varchar │
├─────────┤
│ A,C,G   │
│ A,C     │
│ A,B,C,D │
│ A,B,C   │
│ A,C,F   │
│ A       │
│ A,C,E   │
│ A,S     │
│ A,C,E,L │
└─────────┘

In [125]:
con.sql("""
SELECT
    100*SUM(popn_white)/SUM(popn_total) AS white_pct,
    100*SUM(popn_black)/SUM(popn_total) AS black_pct,
    100*SUM(popn_asian)/SUM(popn_total) AS asian_pct,
    100*SUM(popn_nativ)/SUM(popn_total) AS native_pct,
    SUM(popn_total) AS popn_total
FROM nyc_census_blocks AS census
JOIN nyc_subway_stations AS subways
ON ST_DWithin(census.geom, subways.geom, 200)
WHERE strpos(subways.routes,'A') > 0;
""")

┌───────────────────┬───────────────────┬───────────────────┬────────────────────┬────────────┐
│     white_pct     │     black_pct     │     asian_pct     │     native_pct     │ popn_total │
│      double       │      double       │      double       │       double       │   int128   │
├───────────────────┼───────────────────┼───────────────────┼────────────────────┼────────────┤
│ 45.59012559002023 │ 22.09362356709373 │ 10.43440239379636 │ 0.7923128792987189 │     189824 │
└───────────────────┴───────────────────┴───────────────────┴────────────────────┴────────────┘

In [126]:
con.sql("""
WITH subway_lines AS (
    SELECT DISTINCT trim(unnest(string_split(routes,','))) AS route
    FROM nyc_subway_stations
    WHERE routes is NOT NULL)
SELECT
    lines.route,
    100*SUM(popn_white)/SUM(popn_total) AS white_pct,
    100*SUM(popn_black)/SUM(popn_total) AS black_pct,
    100*SUM(popn_asian)/SUM(popn_total) AS asian_pct,
    100*SUM(popn_nativ)/SUM(popn_total) AS native_pct,
    SUM(popn_total) AS popn_total
FROM nyc_census_blocks AS census
JOIN nyc_subway_stations AS subways
ON ST_DWithin(census.geom, subways.geom, 200)
JOIN subway_lines AS lines
    ON strpos(subways.routes, lines.route) > 0
GROUP BY lines.route
ORDER BY black_pct DESC;
""").df()

,route,white_pct,black_pct,asian_pct,native_pct,popn_total
0,S,39.839644,46.503108,4.510375,0.444431,33301.0
1,3,42.727318,42.061987,6.370855,0.437128,223047.0
2,5,33.793778,41.385627,5.574665,0.839123,218919.0
3,2,39.263049,38.391146,5.827999,0.734757,291661.0
4,C,46.878718,30.598767,7.403826,0.581077,224411.0
5,4,37.553001,27.428313,6.739506,1.173728,174998.0
6,B,39.955882,26.852519,9.166235,0.933421,256583.0
7,A,45.590126,22.093624,10.434402,0.792313,189824.0
8,J,37.629553,21.637651,16.344902,0.935564,132861.0
9,Q,56.884480,20.631412,13.478664,0.367393,127112.0


## National Wetland Analysis

In [127]:
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")

In [128]:
state = "AZ"
url = f"https://data.source.coop/giswqs/nwi/wetlands/{state}_Wetlands.parquet"
con.sql(f"SELECT * FROM '{url}'")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬──────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [129]:
con.sql(f"DESCRIBE FROM '{url}'")

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ ATTRIBUTE    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WETLAND_TYPE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ACRES        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Shape_Length │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Shape_Area   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ geometry     │ GEOMETRY    │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [130]:
con.sql(f"""
SELECT COUNT(*) AS Count
FROM 's3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet'
""")

┌──────────┐
│  Count   │
│  int64   │
├──────────┤
│ 38065251 │
└──────────┘

In [131]:
df = con.sql(f"""
SELECT filename, COUNT(*) AS Count
FROM read_parquet('s3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet', filename=True)
GROUP BY filename
ORDER BY COUNT(*) DESC;
""").df()
df.head()

,filename,Count
0,s3://us-west-2.opendata.source.coop/giswqs/nwi...,2674597
1,s3://us-west-2.opendata.source.coop/giswqs/nwi...,2263259
2,s3://us-west-2.opendata.source.coop/giswqs/nwi...,2077099
3,s3://us-west-2.opendata.source.coop/giswqs/nwi...,1857721
4,s3://us-west-2.opendata.source.coop/giswqs/nwi...,1718865


In [132]:
count_df = con.sql(f"""
SELECT SUBSTRING(filename, LENGTH(filename) - 18, 2) AS State, COUNT(*) AS Count
FROM read_parquet('s3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet', filename=True)
GROUP BY State
ORDER BY COUNT(*) DESC;
""").df()
count_df.head(10)

,State,Count
0,MN,2674597
1,TX,2263259
2,ND,2077099
3,MT,1857721
4,WI,1718865
5,SD,1664305
6,AK,1649192
7,NE,1516060
8,MO,1262141
9,KS,1063234


In [133]:
con.sql("CREATE OR REPLACE TABLE wetlands AS FROM count_df")
con.sql("FROM wetlands")

┌─────────┬─────────┐
│  State  │  Count  │
│ varchar │  int64  │
├─────────┼─────────┤
│ MN      │ 2674597 │
│ TX      │ 2263259 │
│ ND      │ 2077099 │
│ MT      │ 1857721 │
│ WI      │ 1718865 │
│ SD      │ 1664305 │
│ AK      │ 1649192 │
│ NE      │ 1516060 │
│ MO      │ 1262141 │
│ KS      │ 1063234 │
│ ·       │     ·   │
│ ·       │     ·   │
│ ·       │     ·   │
│ NJ      │  248702 │
│ MD      │  240320 │
│ AZ      │  189606 │
│ CT      │  170964 │
│ VT      │  164400 │
│ WV      │  154348 │
│ DE      │   78597 │
│ RI      │   55229 │
│ HI      │   13566 │
│ DC      │    1555 │
├─────────┴─────────┤
│      51 rows      │
│    (20 shown)     │
└───────────────────┘

In [134]:
url = 'https://opengeos.org/data/us/us_states.parquet'
con.sql(
    f"""
CREATE OR REPLACE TABLE states AS
SELECT * FROM '{url}'
"""
)

In [135]:
con.sql("""
SELECT * FROM states INNER JOIN wetlands on states.id = wetlands.State
""")

┌─────────┬───────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [136]:
file_path = "states_with_wetlands.geojson"
con.sql("COPY (SELECT s.name, s.id, w.Count, s.geometry FROM states s JOIN wetlands w ON s.id = w.State ORDER BY w.Count DESC) TO '{file_path}' WITH (FORMAT GDAL, DRIVER 'GeoJSON')")

In [142]:
m = leafmap.Map()
m.add_data(
    file_path, column="Count", scheme="Quantiles", cmap="Greens",
    legend_title="Wetland Count"    
)
m

TypeError: Data must be a GeoDataFrame or a path to a file that can be read by geopandas.read_file().

In [138]:
leafmap.pie_chart(
    count_df, "State", "Count", height=800, title="Number of Wetlands by State"
)

In [139]:
con.sql("SELECT SUM(Shape_Area) / 1000000 AS Area_SqKm FROM 's3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet';")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│     Area_SqKm      │
│       double       │
├────────────────────┤
│ 1442876.9096808902 │
└────────────────────┘

In [140]:
area_df = con.sql("SELECT SUBSTRING(filename, LENGTH(filename) - 18, 2) AS State, SUM(Shape_Area) / 1000000 AS Area_SqKm FROM read_parquet('s3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet', filename=True) GROUP BY State ORDER BY COUNT(*) DESC;").df()
area_df.head(10)

,State,Area_SqKm
0,MN,66376.803485
1,TX,43200.379897
2,ND,17586.222328
3,MT,13916.988281
4,WI,81781.779861
5,SD,14151.357901
6,AK,398576.219916
7,NE,7672.796780
8,MO,11283.318920
9,KS,5748.325735


In [141]:
leafmap.bar_chart(area_df, "State", "Area_SqKm", title="Wetland Area by State")